In [ ]:
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document
import numpy as np
from embedding_deduplication import BailianEmbeddings, process_graph_document, generate_relationship_signature
from typing import List, Tuple

graph_doc = []
unique_nodes: List[Node] = []
unique_relationships: List[Relationship] = []
for gpd in graph_doc:
    for node in gpd.nodes:
        if node not in unique_nodes:
            unique_nodes.append(node)
    for rel in gpd.relationships:
        if rel not in unique_relationships:
            unique_relationships.append(rel)

graph_doc_dedup = GraphDocument(
    source=Document(page_content="合并去重后的图数据", metadata={}),
    nodes=unique_nodes,
    relationships=unique_relationships
)

# 创建文件并写入/data/graph_doc_dedup.pkl
with open("./data/graph_doc_dedup.pkl", "wb") as f:
    import pickle
    pickle.dump(graph_doc_dedup, f)

# with open("/data/graph_doc_dedup.pkl", "wb") as f:
#     import pickle
#     pickle.dump(graph_doc_dedup, f)

node_signatures = [f"({node.type}){node.id}" for node in unique_nodes]
embedded = BailianEmbeddings(api_key="sk-aec12d19a21049a2a56fa32acb855dae")
node_embeddings = embedded.embed_documents(node_signatures)
node_embeddings = np.array(node_embeddings)

from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(node_embeddings)
distance_matrix = 1 - similarity_matrix
distance_matrix = np.maximum(distance_matrix, 0)
threshold = 0.18  # 设置距离阈值

from sklearn.cluster import DBSCAN
dbscan = DBSCAN(eps=threshold, min_samples=1, metric='precomputed')
clusters = dbscan.fit_predict(distance_matrix)

print("节点聚类结果：", clusters)
cluster_graph_doc = {}
for idx, cluster_id in enumerate(clusters):
    if cluster_id not in cluster_graph_doc:
        cluster_graph_doc[cluster_id] = []
    cluster_graph_doc[cluster_id].append(unique_nodes[idx])

for cluster_id, nodes in cluster_graph_doc.items():
    print(f"聚类 {cluster_id} 包含节点：")
    print([f"{node.type}:{node.id}" for node in nodes])
    print("\n")

print(graph_doc_dedup)
print(cluster_graph_doc)
with open("./data/cluster_graph_doc.pkl", "wb") as f:
    import pickle
    pickle.dump(cluster_graph_doc, f)


2025-10-21 20:25:25,746 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:26,473 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:27,106 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:27,772 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:28,464 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:29,101 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:29,799 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2025-10-21 20:25:30,491 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/

节点聚类结果： [ 0  1  2  3  4  5  0  6  7  6  8  9  0  0 10 11  0  0 12 13 14 15  1  4
 16 17 18 18 19 20 21 22 23 24 25 26 24 27 28 29 30 31 32 23 25 23 33 34
 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 53 53
 57 58 59 37 60 60 61 62 63 64 65 45 66 67]
聚类 0 包含节点：
['Network Technology:Circuit switching', 'Network Technology:packet switching', 'Network Technology:Packet switching', 'Network Technology:circuit switching', 'Network Technology:packet switching technology', 'Network Technology:Connection-less packet switching']


聚类 1 包含节点：
['Network Connection:dedicated communications channel', 'Network Connection:dedicated path']


聚类 2 包含节点：
['Network Resource:full bandwidth of the channel']


聚类 3 包含节点：
['Network Resource:channel']


聚类 4 包含节点：
['Network Service:analog telephone networks', 'Network Service:early analogue telephone network']


聚类 5 包含节点：
['Network Technology:message switching']


聚类 6 包含节点：
['Network Channel:separate dedicated signalling channel', 'Netwo

In [ ]:
from transformers import AutoTokenizer, AutoModel, DebertaV2Tokenizer, DebertaV2Model
import torch
model_dir = "/mnt/e/project/DataGraphX_Learn - 副本/bert/deberta-v3-base"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_dir)
model = DebertaV2Model.from_pretrained(pretrained_model_name_or_path=model_dir, output_hidden_states=True)

In [ ]:
def find_positions(tokens, sub_tokens):
    for i in range(len(tokens) - len(sub_tokens) + 1):
        if tokens[i:i+len(sub_tokens)] == sub_tokens:
            # +1 因为模型输出第0位是 [CLS]
            return list(range(i+1, i+1+len(sub_tokens)))
    return []


def get_aggregated_word_vector(hidden_states, indices, num_layers_to_aggregate=4):
    """
    获取一个词在最后几层隐藏状态的聚合向量。
    """
    # 1. 选择最后num_layers_to_aggregate层的向量
    # hidden_states[-1]是最后一层, hidden_states[-4]是倒数第四层
    layers_to_use = hidden_states[-num_layers_to_aggregate:]

    # 2. 对每个选定的层，聚合目标词所有token的向量 (平均法)
    word_vectors_per_layer = []
    for layer_hidden_state in layers_to_use:
        # layer_hidden_state shape: [batch_size, seq_len, hidden_size]
        # 获取目标词所有token的向量
        token_vectors = layer_hidden_state[0, indices, :] # shape: [num_tokens, hidden_size]
        # 对这些token的向量求平均，得到该层中该词的单一表示
        mean_vector = torch.mean(token_vectors, dim=0) # shape: [hidden_size]
        word_vectors_per_layer.append(mean_vector) #    shape: [hidden_size]

    # 3. 聚合来自不同层的词向量 (拼接法或求和/平均法)
    # 拼接法能保留更多信息，但维度会变高
    # final_vector = torch.cat(word_vectors_per_layer, dim=0)
    # 求和/平均法更简单，维度不变
    # final_vector = torch.stack(word_vectors_per_layer, dim=0).sum(dim=0)
    final_vector = torch.mean(torch.stack(word_vectors_per_layer, dim=0), dim=0) # shape: [hidden_size]
    
    return final_vector

In [ ]:
def compute_word_similarity(model:AutoModel, tokenizer:AutoTokenizer, aim_word:str, src_sent:str, rep_sent:str, k:int=4):
    """
    Arguments:
        model: 预训练语言模型
        tokenizer: 预训练语言模型对应的tokenizer
        aim_word: 替换句子中的词
        src_sent: 包含aim_word的源句子
        rep_sent: 包含aim_word的替换句子
        k: 聚合最后k层的隐藏状态
    Returns:
        cos_similarity: 基于最后k层隐藏状态的余弦相似度
        emb_cos_similarity: 基于embedding层的余弦相似度
    """
    # 对源句子和替换句子进行tokenize
    src_input = tokenizer(src_sent, return_tensors="pt")
    rep_input = tokenizer(rep_sent, return_tensors="pt")

    # 获取源词和替换词的token ids
    src_sent_tokens = tokenizer.tokenize(src_sent)
    rep_sent_tokens = tokenizer.tokenize(rep_sent)
    aim_word_tokens = tokenizer.tokenize(aim_word)
    print("源句子tokens:", src_sent_tokens)
    print("替换句子tokens:", rep_sent_tokens)
    print("目标词tokens:", aim_word_tokens)
    # src_word_tokens = tokenizer.tokenize(src_word)
    # rep_word_tokens = tokenizer.tokenize(rep_word)

    pos_src = find_positions(src_sent_tokens, aim_word_tokens)
    pos_rep = find_positions(rep_sent_tokens, aim_word_tokens)
    print("源句子中目标词位置:", pos_src)
    print("替换句子中目标词位置:", pos_rep)

    # 获取模型输出的隐藏状态
    with torch.no_grad():
        src_outputs = model(**src_input)
        rep_outputs = model(**rep_input)

    # 聚合最后k层的词向量并计算余弦相似度
    src_last_k_layers = get_aggregated_word_vector(src_outputs.hidden_states, pos_src, num_layers_to_aggregate=k)
    rep_last_k_layers = get_aggregated_word_vector(rep_outputs.hidden_states, pos_rep, num_layers_to_aggregate=k)
    cos_similarity = torch.nn.functional.cosine_similarity(src_last_k_layers, rep_last_k_layers, dim=0) # 基于最后k层隐藏状态的余弦相似度

    src_embedding = src_outputs.hidden_states[0]
    rep_embedding = rep_outputs.hidden_states[0]
    src_emb_vec = src_embedding[0, pos_src, :]
    rep_emb_vec = rep_embedding[0, pos_rep, :]
    mean_src_emb_vec = torch.mean(src_emb_vec, dim=0)
    mean_rep_emb_vec = torch.mean(rep_emb_vec, dim=0)
    emb_cos_similarity = torch.nn.functional.cosine_similarity(mean_src_emb_vec, mean_rep_emb_vec, dim=0) # 基于embedding层的余弦相似度
    # print("embedding余弦相似度:", emb_cos_similarity.item())

    return cos_similarity.item(), emb_cos_similarity.item()

In [ ]:
def generate_relationship_signature(rel: Relationship) -> str:
    return f"{rel.source.type} {rel.source.id} {rel.type} {rel.target.type} {rel.target.id}"
def generate_node_signature(node: Node) -> str:
    return f"{node.type} {node.id}"
def if_need_merge(node1: Node, node2: Node, graph: GraphDocument, model:AutoModel, tokenizer:AutoTokenizer, k:int) -> bool:
    # 从graph中找到与node1，node2相关的所有关系
    
    related_rels_node1 = [rel for rel in graph.relationships if rel.source == node1 or rel.target == node1]
    related_rels_node2 = [rel for rel in graph.relationships if rel.source == node2 or rel.target == node2]
    print(f"node1{generate_node_signature(node1)}相关关系：", [generate_relationship_signature(rel) for rel in related_rels_node1])
    print(f"node2{generate_node_signature(node2)}相关关系：", [generate_relationship_signature(rel) for rel in related_rels_node2])

    node1_signature = generate_node_signature(node1)
    node2_signature = generate_node_signature(node2)
    computed_rels = []
    for rel1 in related_rels_node1:
        for rel2 in related_rels_node2:
            if rel1 == rel2:
                print("相同关系，跳过比较")
                continue
            rel1_signature = generate_relationship_signature(rel1)
            rel2_signature = generate_relationship_signature(rel2)
            src_word = node2_signature
            aim_word = node1_signature

            src_sent = rel1_signature # node1的源关系
            rep_sent = rel2_signature.replace(src_word, aim_word) # 将node2的源关系中的node2替换为node1
            print("比较关系：", src_sent,"  ", rep_sent)
            cos_similarity, embedding_similarity = compute_word_similarity(src_sent=src_sent, rep_sent=rep_sent, aim_word=aim_word, model=model, tokenizer=tokenizer, k=k)
            print(f"关系相似度: {cos_similarity}, embedding相似度: {embedding_similarity}")



In [ ]:
node1 = cluster_graph_doc[1][0]
node2 = cluster_graph_doc[1][1]
if_need_merge(node1, node2, graph_doc_dedup, model, tokenizer, 8)

node1Network Connection dedicated communications channel相关关系： ['Network Technology Circuit switching establishes Network Connection dedicated communications channel']
node2Network Connection dedicated path相关关系： ['Network Technology Circuit switching requires Network Connection dedicated path']
比较关系： Network Technology Circuit switching establishes Network Connection dedicated communications channel    Network Technology Circuit switching requires Network Connection dedicated communications channel
源句子tokens: ['▁Network', '▁Technology', '▁Circuit', '▁switching', '▁establishes', '▁Network', '▁Connection', '▁dedicated', '▁communications', '▁channel']
替换句子tokens: ['▁Network', '▁Technology', '▁Circuit', '▁switching', '▁requires', '▁Network', '▁Connection', '▁dedicated', '▁communications', '▁channel']
目标词tokens: ['▁Network', '▁Connection', '▁dedicated', '▁communications', '▁channel']
源句子中目标词位置: [6, 7, 8, 9, 10]
替换句子中目标词位置: [6, 7, 8, 9, 10]
关系相似度: 0.9825155735015869, embedding相似度: 1.000000119

In [ ]:
test1 = "apple is a kind of fruit."
test2 = "apple is a tech company."
test3 = "apple is rich in vitamin."
test4 = "apple company has produced iPhone."
test5 = "I went to the bank to deposit money."
test6 = "The river bank was flooded after the heavy rain."

test7 = "vector is a geometric object that has both magnitude and direction, and satisfies the parallelogram law."
"""向量是同时具有数值和方向，且满足平行四边形法则的几何对象"""
test8 = "tensor is a fundamental data structure in machine learning, representing multi-dimensional arrays that can express scalars, vectors, matrices, and higher-dimensional data."
"""张量是机器学习中的一种基本数据结构，它是一个多维数组，可以表示标量、向量、矩阵以及更高维度的数据。"""

aim = "apple"
src = "apple"

src_sent = test2
rep_sent = test4.replace(src, aim)
k = 8
cos_similarity, emb_cos_similarity = compute_word_similarity(model=model, tokenizer=tokenizer, aim_word=aim, src_sent=src_sent, rep_sent=rep_sent, k=k)
print(f"使用最后{k}层隐藏状态聚合的余弦相似度: {cos_similarity}")
print(f"使用embedding层的余弦相似度: {emb_cos_similarity}")


使用最后8层隐藏状态聚合的余弦相似度: 0.8391099572181702
使用embedding层的余弦相似度: 0.9999998807907104


In [ ]:
from transformers import BertTokenizer, BertModel
import torch
model_dir = "/mnt/e/project/DataGraphX_Learn - 副本/bert/models/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44"

In [ ]:
# 新增：比较替换前后指定词的 BERT 向量
from transformers import AutoTokenizer, AutoModel
from transformers import BertTokenizer, BertModel
import torch

# 加载本地模型
model_dir = "/mnt/e/project/DataGraphX_Learn - 副本/bert/models/models--hfl--chinese-roberta-wwm-ext/snapshots/5c58d0b8ec1d9014354d691c538661bf00bfdb44"
tokenizer = BertTokenizer.from_pretrained(model_dir)
model = BertModel.from_pretrained(pretrained_model_name_or_path=model_dir, output_hidden_states=True)

# 原始句子和替换后的句子
test1 = "网络层提供了将数据包从一个节点传输到“不同网络”中连接的另一个节点的功能"
test2 = "苹果是一家公司"
test3 = "向量是同时具有数值和方向，且满足平行四边形法则的几何对象"
test4 = "张量是机器学习中的一种基本数据结构，它是一个多维数组，可以表示标量、向量、矩阵以及更高维度的数据。"
test5 = "张量是将向量和矩阵推广到任意维度的泛化形式"
orig_compare_sent = test4
orig_sent = test3
target = "向量"
replacement = "张量"

aim = "苹果"

mod_sent = orig_sent.replace(target, replacement)


enc_orig = tokenizer(orig_compare_sent, return_tensors="pt")
enc_mod  = tokenizer(mod_sent, return_tensors="pt")
# print("原始句子编码:", enc_orig)
# print("替换后句子编码:", enc_mod)

# # 前向计算 last_hidden_state
with torch.no_grad():
    last_orig = model(**enc_orig).last_hidden_state  # [1, seq_len, hidden]
    last_mod  = model(**enc_mod).last_hidden_state
    pool_orig = model(**enc_orig).pooler_output
    pool_mod  = model(**enc_mod).pooler_output

# # 辅助：在 token 列表中定位子词位置（跳过 [CLS]）
def find_positions(tokens, sub_tokens):
    for i in range(len(tokens) - len(sub_tokens) + 1):
        if tokens[i:i+len(sub_tokens)] == sub_tokens:
            # +1 因为模型输出第0位是 [CLS]
            return list(range(i+1, i+1+len(sub_tokens)))
    return []

tokens_orig = tokenizer.tokenize(orig_compare_sent)
tokens_mod  = tokenizer.tokenize(mod_sent)
sub_orig    = tokenizer.tokenize(replacement)
sub_mod     = tokenizer.tokenize(replacement)
# tokens_aim = tokenizer.tokenize(aim)

print("原始句子 tokens:", tokens_orig)
print("替换后句子 tokens:", tokens_mod)
print("目标子词 tokens:", sub_orig)
print("替换子词 tokens:", sub_mod)

pos_orig = find_positions(tokens_orig, sub_orig)
pos_mod  = find_positions(tokens_mod, sub_mod)

print("原始句子中目标子词位置:", pos_orig)
print("替换后句子中目标子词位置:", pos_mod)

# # 提取对应位置的向量
vec_orig = last_orig[0, pos_orig, :]  # shape = [len(sub_orig), hidden]
vec_mod  = last_mod[0, pos_mod, :]    # shape = [len(sub_mod), hidden]

print("原句中“%s”的向量:" % replacement, vec_orig)
print("替换后“%s”的向量:" % replacement, vec_mod)


# # 计算向量均值（如果子词被拆分成多个，则取均值作为整体表示）
mean_orig = vec_orig.mean(dim=0)  # shape = [hidden]
mean_mod  = vec_mod.mean(dim=0)

# # 计算余弦相似度
cos_sim = torch.nn.functional.cosine_similarity(mean_orig, mean_mod, dim=0)
print("替换前后两词的向量余弦相似度:", cos_sim.item())

# 分别计算每个字之间的相似度
cos_char_vector = []
for i in range(vec_orig.shape[0]):
    for j in range(vec_mod.shape[0]):
        # if target[i] == replacement[j]:
        #     continue
        cos_char = torch.nn.functional.cosine_similarity(vec_orig[i], vec_mod[j], dim=0)
        cos_char_vector.append(cos_char.item())
        print("原句中“%s”与替换后“%s”的向量余弦相似度: %.4f" % (tokens_orig[pos_orig[i]-1], tokens_mod[pos_mod[j]-1], cos_char.item()))

average_cos_char = sum(cos_char_vector) / len(cos_char_vector) if cos_char_vector else 0.0
print("原句中“%s”与替换后“%s”的平均向量余弦相似度: %.4f" % (replacement, replacement, average_cos_char))

cos_sen = torch.nn.functional.cosine_similarity(pool_orig.squeeze(0), pool_mod.squeeze(0), dim=0)
# print("句向量形状:", pool_orig.shape, pool_mod.shape)
print("替换前后两句的句向量余弦相似度:", cos_sen.item())

原始句子 tokens: ['张', '量', '是', '机', '器', '学', '习', '中', '的', '一', '种', '基', '本', '数', '据', '结', '构', '，', '它', '是', '一', '个', '多', '维', '数', '组', '，', '可', '以', '表', '示', '标', '量', '、', '向', '量', '、', '矩', '阵', '以', '及', '更', '高', '维', '度', '的', '数', '据', '。']
替换后句子 tokens: ['张', '量', '是', '同', '时', '具', '有', '数', '值', '和', '方', '向', '，', '且', '满', '足', '平', '行', '四', '边', '形', '法', '则', '的', '几', '何', '对', '象']
目标子词 tokens: ['张', '量']
替换子词 tokens: ['张', '量']
原始句子中目标子词位置: [1, 2]
替换后句子中目标子词位置: [1, 2]
原句中“张量”的向量: tensor([[ 1.1652,  0.1915, -0.0212,  ..., -0.1792, -0.2564, -0.1659],
        [ 0.6932, -0.4313,  0.3734,  ..., -0.4646, -0.8581, -0.2276]])
替换后“张量”的向量: tensor([[ 1.0185,  0.1850, -0.1780,  ..., -0.4764, -0.4025, -0.5814],
        [ 0.9244, -0.4699, -0.0493,  ..., -0.4265, -0.6933, -0.4978]])
替换前后两词的向量余弦相似度: 0.9046716690063477
原句中“张”与替换后“张”的向量余弦相似度: 0.9095
原句中“张”与替换后“量”的向量余弦相似度: 0.6376
原句中“量”与替换后“张”的向量余弦相似度: 0.5483
原句中“量”与替换后“量”的向量余弦相似度: 0.8961
原句中“张量”与替换后“张量”的平均向量余弦相似度: 0.7479
替换

In [ ]:
with open("./data/cluster_graph_doc.pkl", "rb") as f: 
    import pickle 
    cluster_graph_doc = pickle.load(f)

print(cluster_graph_doc.keys())
print(cluster_graph_doc.items())

with open("./data/graph_doc_dedup.pkl", "rb") as f: 
    import pickle 
    graph_doc_dedup = pickle.load(f)



dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72])
dict_items([(0, [Node(id='Fast Ethernet', type='Network Standard', properties={}), Node(id='10-megabit Ethernet', type='Network Standard', properties={})]), (1, [Node(id='IEEE 802.3u', type='Network Standard', properties={}), Node(id='IEEE 802.12', type='Network Standard', properties={}), Node(id='IEEE 802.3', type='Network Standard', properties={}), Node(id='IEEE 802.11', type='Network Standard', properties={})]), (2, [Node(id='CSMA/CD', type='Protocol', properties={})]), (3, [Node(id='100BASE-TX', type='Network Model', properties={}), Node(id='10BASE-T', type='Network Model', properties={}), Node(id='100BASE-T4', type='Network Model', properties={}), Node(id='100BASE-T2', type='Network Model', p